# Alzheimer's Disease Diagnosis Prediction
## Model: K-Nearest Neighbors (KNN)
**Target:** `Diagnosis` — 0 = No Alzheimer's, 1 = Alzheimer's

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt
import seaborn as sns

## 2. Load & Prepare Data

In [ ]:
df = pd.read_csv('../datsets/alzheimers_disease_data.csv')

# Drop non-predictive identifier columns
df = df.drop(columns=['PatientID', 'DoctorInCharge'])

print(f"Dataset shape: {df.shape}")
print(f"\nTarget distribution:")
print(df['Diagnosis'].value_counts().rename({0: 'No AD (0)', 1: 'AD (1)'}))
print(f"\nMissing values: {df.isnull().sum().sum()}")
df.head()

## 3. Split Features and Target

In [ ]:
X = df.drop(columns=['Diagnosis'])
y = df['Diagnosis']

# 80/20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples : {X_train.shape[0]}")
print(f"Test samples     : {X_test.shape[0]}")

## 4. Feature Scaling
KNN is a distance-based algorithm — standardization is essential so no single feature dominates the distance calculation.

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print("Features scaled with StandardScaler (mean=0, std=1).")

## 5. Find Optimal K
Evaluate odd values of K from 1–25 to avoid ties and identify the best number of neighbors.

In [ ]:
k_values = list(range(1, 26, 2))  # odd values: 1, 3, 5, ..., 25
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = []
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    score = cross_val_score(knn, X_train_sc, y_train, cv=cv, scoring='roc_auc').mean()
    cv_scores.append(score)

best_k = k_values[np.argmax(cv_scores)]
print(f"Best K : {best_k}  (CV AUC = {max(cv_scores):.4f})")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(k_values, cv_scores, marker='o', color='#C44E52', linewidth=2)
ax.axvline(best_k, color='#555555', linestyle='--', linewidth=1.5, label=f'Best K = {best_k}')
ax.set_xlabel('Number of Neighbors (K)', fontsize=12)
ax.set_ylabel('CV ROC-AUC', fontsize=12)
ax.set_title('KNN — Optimal K Selection', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Hyperparameter Tuning (GridSearchCV)

In [ ]:
param_grid = {
    'n_neighbors': list(range(1, 26, 2)),
    'weights':     ['uniform', 'distance'],
    'metric':      ['euclidean', 'manhattan']
}

grid_search = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_sc, y_train)

print(f"\nBest parameters : {grid_search.best_params_}")
print(f"Best CV AUC     : {grid_search.best_score_:.4f}")

## 7. Train Best KNN Model

In [ ]:
best_knn = grid_search.best_estimator_

# Cross-validation scores on training set
cv_acc = cross_val_score(best_knn, X_train_sc, y_train, cv=cv, scoring='accuracy')
cv_auc = cross_val_score(best_knn, X_train_sc, y_train, cv=cv, scoring='roc_auc')

print("5-Fold Cross-Validation (Training Set)")
print(f"  Accuracy : {cv_acc.mean():.4f}  ±  {cv_acc.std():.4f}")
print(f"  ROC-AUC  : {cv_auc.mean():.4f}  ±  {cv_auc.std():.4f}")

## 8. Evaluate on Test Set

In [ ]:
y_pred  = best_knn.predict(X_test_sc)
y_proba = best_knn.predict_proba(X_test_sc)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print("=" * 45)
print("  KNN — Test Set Results")
print("=" * 45)
print(f"  Accuracy  : {acc:.4f}")
print(f"  ROC-AUC   : {auc:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=['No AD (0)', 'AD (1)']))

## 9. Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No AD', 'AD'])
disp.plot(cmap='Reds', colorbar=False, ax=ax)

ax.set_title('KNN — Confusion Matrix', fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

## 10. ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color='#C44E52', linewidth=2.5,
        label=f'KNN (AUC = {auc:.3f})')
ax.plot([0, 1], [0, 1], '--', color='gray', linewidth=1.5, label='Random Chance')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('KNN — ROC Curve', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Results Summary

In [ ]:
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
precision   = tp / (tp + fp)
f1          = 2 * precision * sensitivity / (precision + sensitivity)

summary = pd.DataFrame({
    'Metric': ['Accuracy', 'ROC-AUC', 'Sensitivity (Recall)', 'Specificity',
               'Precision', 'F1-Score', 'CV Accuracy (mean ± std)',
               'CV AUC (mean ± std)', 'Best K', 'Best Weights', 'Best Metric'],
    'Value':  [
        f'{acc:.4f}', f'{auc:.4f}',
        f'{sensitivity:.4f}', f'{specificity:.4f}',
        f'{precision:.4f}', f'{f1:.4f}',
        f'{cv_acc.mean():.4f} ± {cv_acc.std():.4f}',
        f'{cv_auc.mean():.4f} ± {cv_auc.std():.4f}',
        str(grid_search.best_params_['n_neighbors']),
        grid_search.best_params_['weights'],
        grid_search.best_params_['metric']
    ]
})

print(summary.to_string(index=False))